# Lesson 8: Gradient Boosting

Sequential ensembles that learn from mistakes.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.datasets import load_breast_cancer, make_regression
from sklearn.tree import DecisionTreeRegressor

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

## 1. Visualizing Sequential Residual Fitting

In [ ]:
X = np.sort(np.random.rand(80, 1) * 10, axis=0)
y = np.sin(X).ravel() + np.random.normal(0, 0.15, X.shape[0])

F = np.zeros_like(y)
fig, axes = plt.subplots(1, 5, figsize=(20, 3))

for i in range(5):
    residual = y - F
    tree = DecisionTreeRegressor(max_depth=3)
    tree.fit(X, residual)
    F += 0.5 * tree.predict(X)
    axes[i].scatter(X, y, alpha=0.3)
    axes[i].scatter(X, F, color='red', s=20)
    axes[i].set_title(f'Stage {i+1}')
    axes[i].set_ylim(-1.5, 1.5)

plt.suptitle('Boosting: Sequential Residual Fitting')
plt.tight_layout()
plt.show()

## 2. Effect of Learning Rate

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

learning_rates = [0.01, 0.05, 0.1, 0.5, 1.0]
train_scores, test_scores = [], []

for lr in learning_rates:
    gb = GradientBoostingClassifier(
        n_estimators=100, learning_rate=lr, max_depth=3, random_state=42
    )
    gb.fit(X_train, y_train)
    train_scores.append(gb.score(X_train, y_train))
    test_scores.append(gb.score(X_test, y_test))
    print(f"LR={lr:.2f}: Train={train_scores[-1]:.3f}, Test={test_scores[-1]:.3f}")

plt.figure(figsize=(8, 5))
plt.plot(learning_rates, train_scores, 'o-', label='Train')
plt.plot(learning_rates, test_scores, 's-', label='Test')
plt.xlabel('Learning Rate')
plt.ylabel('Accuracy')
plt.title('Effect of Learning Rate')
plt.xscale('log')
plt.legend()
plt.grid(True)
plt.show()

## 3. Learning Curves (Accuracy vs. n_estimators)

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=3,
    subsample=0.8, random_state=42
)
gb.fit(X_train, y_train)

# Track staged predictions
train_staged = list(gb.staged_score(X_train, y_train))
test_staged = list(gb.staged_score(X_test, y_test))

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_staged)+1), train_staged, label='Train')
plt.plot(range(1, len(test_staged)+1), test_staged, label='Test')
plt.xlabel('Number of Trees')
plt.ylabel('Accuracy')
plt.title('Learning Curves')
plt.legend()
plt.grid(True)
plt.show()

## 4. Feature Importance

In [ ]:
importance = pd.DataFrame({
    'feature': data.feature_names,
    'importance': gb.feature_importances_
}).sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(10, 5))
plt.barh(importance['feature'], importance['importance'])
plt.xlabel('Importance')
plt.title('Gradient Boosting Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Gradient Boosting Regressor

In [ ]:
X_r, y_r = make_regression(n_samples=500, n_features=10, noise=20, random_state=42)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X_r, y_r, test_size=0.2)

gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3)
gbr.fit(Xr_tr, yr_tr)
print(f"Train R²: {gbr.score(Xr_tr, yr_tr):.3f}")
print(f"Test R²: {gbr.score(Xr_te, yr_te):.3f}")

## 6. Biotechnology Example: Drug Binding

In [ ]:
n = 500
drug = pd.DataFrame({
    'mol_weight': np.random.normal(400, 100, n),
    'logP': np.random.uniform(-2, 5, n),
    'h_donors': np.random.poisson(2, n),
    'h_acceptors': np.random.poisson(4, n),
})
score = 0.3*drug['logP'] - 0.1*drug['mol_weight']/100 + 0.2*drug['h_acceptors'] + np.random.normal(0,0.2,n)
drug['binds'] = (score > score.median()).astype(int)

gb_drug = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb_drug.fit(drug.drop('binds', axis=1), drug['binds'])
for name, imp in zip(drug.columns[:-1], gb_drug.feature_importances_):
    print(f"{name}: {imp:.3f}")

## 7. Exercises

### Level 1
What is the key difference between bagging (RF) and boosting (GB)?

### Level 2
Train GB on breast cancer with learning_rates [0.01, 0.05, 0.1, 0.5], n_estimators=100. Plot test accuracy vs. learning rate.

### Level 3
You have 50 samples and 200 features. Why might boosting be risky?

In [ ]:
# Level 2 code

## 8. Coding Challenge

Write `tune_gradient_boosting(X_train, y_train, X_val, y_val)` with grid search over learning_rate and max_depth.

In [ ]:
def tune_gradient_boosting(X_train, y_train, X_val, y_val):
    best_model, best_acc = None, 0
    for lr in [0.01, 0.05, 0.1]:
        for d in [2, 3, 5]:
            gb = GradientBoostingClassifier(
                n_estimators=100, learning_rate=lr, max_depth=d, random_state=42
            )
            gb.fit(X_train, y_train)
            acc = accuracy_score(y_val, gb.predict(X_val))
            print(f"LR={lr}, depth={d}: val={acc:.3f}")
            if acc > best_acc:
                best_acc, best_model = acc, gb
    print(f"Best: {best_acc:.3f}")
    return best_model

X_tr, X_va, y_tr, y_va = train_test_split(X_train, y_train, test_size=0.25, random_state=42)
best_gb = tune_gradient_boosting(X_tr, y_tr, X_va, y_va)

## Summary

- Boosting trains trees sequentially, each correcting previous errors
- Each tree fits residuals (negative gradient)
- Low learning rate + many trees = high performance
- Shallow trees (depth 2-5) work best
- XGBoost/LightGBM are production-ready implementations
- Use early stopping to prevent overfitting